### Q2A1 Min Cost 

#### Original - 1b

In [28]:
import gurobipy as gp
from gurobipy import GRB

b = {'a': 12, 
     'b': 6, 
     'c': 0, 
     'e': -6, 
     'f': -1, 
     'd': -11}

arcs = {
    ('a', 'b'): (10, 5),
    ('a', 'e'): (70, 2),
    ("a", "d"): (100, 9),
    ("b", "c"): (40, 5),
    ("b", "e"): (80, 8),
    ("c", "e"): (60, 7),
    ("c", "f"): (20, 15),
    ("d", "c"): (-60, 4),
    ("e", "f"): (10, 9),
    ("f", "d"): (30, 10),
}

m = gp.Model("MinCostFlow")
x = m.addVars(arcs.keys(), name="flow", lb=0)
m.addConstrs((x[i, j] <= arcs[i, j][1] for i, j in arcs), name="cap")
m.setObjective(gp.quicksum(arcs[i, j][0] * x[i, j] for i, j in arcs), GRB.MINIMIZE)
m.addConstrs(
    (
        gp.quicksum(x[v, j] for j in b if (v, j) in x)
        - gp.quicksum(x[i, v] for i in b if (i, v) in x)
        == b[v]
        for v in b
    ),
    name="balance",
)
m.update()
m.write("a2q1_mincost_b.lp")
m.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.0.0 25A354)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 16 rows, 10 columns and 30 nonzeros
Model fingerprint: 0x6046bed1
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+01, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 2e+01]
Presolve removed 11 rows and 1 columns
Presolve time: 0.00s
Presolved: 5 rows, 9 columns, 15 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    6.6913000e+02   1.201050e+01   0.000000e+00      0s
       7    1.5700000e+03   0.000000e+00   0.000000e+00      0s

Solved in 7 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.570000000e+03


In [29]:
m.printAttr("x")


    Variable            x 
-------------------------
   flow[a,b]            3 
   flow[a,e]            2 
   flow[a,d]            7 
   flow[b,c]            5 
   flow[b,e]            4 
   flow[c,f]            9 
   flow[d,c]            4 
   flow[f,d]            8 


#### Fractional - 1c

In [30]:
b = {
    'a':  12,
    'b':   6,
    'c':   0,
    'e':  -6,
    'f':  -1,
    'd': -11,
}

arc_data = {
    ('a', 'b'): (10,  5),
    ('a', 'e'): (70,  2),
    ('a', 'd'): (100, 9),
    ('b', 'c'): (40,  5),
    ('b', 'e'): (80,  8),
    ('c', 'e'): (60,  7),
    ('c', 'f'): (20, 15),
    ('d', 'c'): (-60, 4),
    ('e', 'f'): (10,  9),
    ('f', 'd'): (30, 10),
}

arc_data[('b','c')] = (arc_data[('b','c')][0], 5.5)  # increase cap (see my note)
arc_data[('a','d')] = (arc_data[('a','d')][0], 6.5)  # decrease cap (see my note)

arcs = list(arc_data.keys())
cost = {a: arc_data[a][0] for a in arcs}
cap  = {a: arc_data[a][1] for a in arcs}

m = gp.Model("MinCostFlow")
x = m.addVars(arcs, name="flow", lb=0, ub=cap)
m.setObjective(gp.quicksum(cost[a] * x[a] for a in arcs), GRB.MINIMIZE)

V = list(b.keys())
for v in V:
    outflow = gp.quicksum(x[i,j] for (i,j) in arcs if i == v)
    inflow  = gp.quicksum(x[i,j] for (i,j) in arcs if j == v)
    m.addConstr(outflow - inflow == b[v], name=f"bal[{v}]")

m.update()
m.write("a2q1_mincost_c.lp")
m.optimize()

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (mac64[arm] - Darwin 25.0.0 25A354)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 6 rows, 10 columns and 20 nonzeros
Model fingerprint: 0x3724e1d7
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+01, 1e+02]
  Bounds range     [2e+00, 2e+01]
  RHS range        [1e+00, 1e+01]
Presolve time: 0.00s
Presolved: 6 rows, 10 columns, 20 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0   -1.2000000e+02   5.100000e+01   0.000000e+00      0s
       6    1.5700000e+03   0.000000e+00   0.000000e+00      0s

Solved in 6 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.570000000e+03


In [31]:
m.printAttr("x")


    Variable            x 
-------------------------
   flow[a,b]          3.5 
   flow[a,e]            2 
   flow[a,d]          6.5 
   flow[b,c]          5.5 
   flow[b,e]            4 
   flow[c,f]          9.5 
   flow[d,c]            4 
   flow[f,d]          8.5 
